In [1]:
pip install optuna optuna-dashboard

Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
import glob
import random
import math
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import train_test_split
import optuna

# Disable Weights & Biases to prevent spamming your dashboard with 50 trials
import wandb
wandb.init(mode="disabled")

# ==========================================
# 1. GLOBAL SETUP & SEED
# ==========================================
def set_seed(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    print(f"✅ Global seed set to: {seed}")

set_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using Device: {device}")

/home/psxkf4/miniconda3/envs/phypush/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✅ Global seed set to: 42
Using Device: cuda


In [3]:
# ==========================================
# 2. CONFIGURATION & PATHS
# ==========================================
CSV_PATH = "/home/psxkf4/IsaacLab/source/collected_data/data_tb-3_ta57_emavel1.0_velstd0.0.csv"
OFFLINE_DATA_DIR = "/home/psxkf4/panda_phypush/csv_data/offline_collection"
BEST_MODEL_SAVE_PATH = "./best_optuna_model.pth"

SEQ_LEN = 60
BATCH_SIZE = 64
TUNE_EPOCHS = 1000  # Reduced from 1000 for faster HPO iterations

G = 9.81
GLOBAL_M_RANGE = 1.0 - 0.3
GLOBAL_MU_RANGE = 0.5 - 0.3
SMOOTHING_WINDOW_SIZE = 3

In [4]:
# ==========================================
# 3. DATA PREPARATION (DONE ONCE)
# ==========================================
print("\n--- Loading Simulation Data ---")
df = pd.read_csv(CSV_PATH)

def clean_force_col(x):
    if isinstance(x, str): return float(x.strip('[]'))
    return x

if 'gt_fric_force' in df.columns:
    df['gt_fric_force'] = df['gt_fric_force'].apply(clean_force_col)

# Domain Filtering
m_seen_max, m_seen_min = 1.0, 0.3
mu_seen_max, mu_seen_min = 0.5, 0.3

conditions = [
    (df['gt_mass'] >= m_seen_min) & (df['gt_mass'] <= m_seen_max) & 
    (df['gt_mu'] >= mu_seen_min) & (df['gt_mu'] <= mu_seen_max)
]
df['domain'] = np.select(conditions, ['m_seen_mu_seen'], default='other')
df_filtered = df[df['domain'] == 'm_seen_mu_seen'].copy()

print(f"Filtered Dataset: {len(df_filtered)} samples.")

# Extract Columns
acc_cols = sorted([c for c in df_filtered.columns if "input_acc_" in c], key=lambda x: int(x.split('_')[-1]))
vel_cols = sorted([c for c in df_filtered.columns if "input_vel_" in c], key=lambda x: int(x.split('_')[-1]))

X_acc_flat = df_filtered[acc_cols].values.astype(np.float32)
X_vel_flat = df_filtered[vel_cols].values.astype(np.float32)
y = df_filtered[['gt_mass', 'gt_mu']].values.astype(np.float32)

X_acc = X_acc_flat.reshape(-1, SEQ_LEN, 1)
X_vel = X_vel_flat.reshape(-1, SEQ_LEN, 1)

# Tensors
X_acc_tensor = torch.tensor(X_acc)
X_vel_tensor = torch.tensor(X_vel)
y_tensor = torch.tensor(y)

# Extract PINN Tensors
robot_fz_list, rhs_acc_list, lhs_net_f_list, table_fz_list, robot_fx_list = [], [], [], [], []
for idx, row in df_filtered.iterrows():
    st = int(row['start_t'])
    window_range = range(st, st + SEQ_LEN)
    robot_fz_list.append([row[f"pinn_robot_wrench_t{t}_ax5"] for t in window_range])
    rhs_acc_list.append([row[f"pinn_RHS_acc_t{t}_ax3"] for t in window_range])
    lhs_net_f_list.append([row[f"pinn_LHS_wrench_t{t}_ax3"] for t in window_range])
    table_fz_list.append([row[f"pinn_table_wrench_t{t}_ax5"] for t in window_range])
    robot_fx_list.append([row[f"pinn_robot_wrench_t{t}_ax3"] for t in window_range])

fz_robot_tensor = torch.tensor(np.array(robot_fz_list), dtype=torch.float32)
rhs_acc_tensor = torch.tensor(np.array(rhs_acc_list), dtype=torch.float32)
lhs_net_f_tensor = torch.tensor(np.array(lhs_net_f_list), dtype=torch.float32)
fz_normal_tensor = torch.tensor(np.array(table_fz_list), dtype=torch.float32)
fx_robot_tensor = torch.tensor(np.array(robot_fx_list), dtype=torch.float32)
start_t_tensor = torch.tensor(df_filtered['start_t'].values.astype(np.int64))

# Split & Loaders
indices = np.arange(len(df_filtered))
train_idx, val_idx = train_test_split(indices, test_size=0.2, random_state=42)

def create_dataset(idx_list):
    return TensorDataset(
        X_acc_tensor[idx_list], X_vel_tensor[idx_list], y_tensor[idx_list],
        fz_robot_tensor[idx_list], rhs_acc_tensor[idx_list], lhs_net_f_tensor[idx_list],
        fz_normal_tensor[idx_list], start_t_tensor[idx_list], fx_robot_tensor[idx_list]
    )

train_loader = DataLoader(create_dataset(train_idx), batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(create_dataset(val_idx), batch_size=BATCH_SIZE, shuffle=False)


--- Loading Simulation Data ---
Filtered Dataset: 5031 samples.


In [5]:
# ==========================================
# 4. MODEL & LOSS CLASSES
# ==========================================
def log_mse_loss(pred, target):
    log_pred = torch.log1p(torch.abs(pred))
    log_target = torch.log1p(torch.abs(target))
    return F.mse_loss(log_pred, log_target)

In [6]:
class PositionalEncoding(nn.Module):
    """
    MATHEMATICAL LOGIC:
    Since Transformers process sequences in parallel, they lack an inherent sense of time.
    We inject a sinusoidal signal PE(pos, 2i) = sin(pos / 10000^(2i/d_model)) to encode 
    temporal order. This allows the model to distinguish between an acceleration spike 
    at t=65 versus steady-state motion at t=80.
    """
    def __init__(self, d_model, max_len=500):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        
        # div_term defines the wavelength of the sinusoidal signals.
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe.unsqueeze(0))

    def forward(self, x):
        # Math: z_t = embedding_vector_t + PE_vector_t
        return x + self.pe[:, :x.size(1), :]

class PhysicsTransformerEstimator(nn.Module):
    def __init__(
        self,
        input_dim=1,         # Subscript x_t: [vel]
        d_model=64,          # Latent dimension (d)
        nhead=4,
        # =========================================================================================
        # MULTI-HEAD ATTENTION (nhead=4)
        # =========================================================================================
        # 1. SPLITTING: The d_model (64) is split into 4 heads of 16 dimensions each.
        #    Math: head_dim = d_model // nhead = 16.
        #
        # 2. PARALLEL PHYSICS: Each head has its own W_Q, W_K, and W_V matrices.
        #    This allows Head 1 to spotlight the 'Spike' while Head 2 spotlights the 'Slide'.
        #
        # 3. CONCATENATION: The 4 outputs [Batch, 20, 16] are glued together to form [Batch, 20, 64].
        #    Final Context = Concat(head_1, head_2, head_3, head_4) @ W_O.
        #
        # 4. WHY 4?: It provides enough diversity to capture mass (transient) and 
        #    friction (steady-state) without making the feature space too small.
        # =========================================================================================
        num_encoder_layers=2, 
        dim_feedforward=128, 
        seq_len=60, 
        dropout=0.1,
        sharpness=10.0,
        cross_sharpness=10.0,
        m_sharpness=30.0,
        mu_sharpness=10.0,
        version=1,
        max_mass_scale=3.0,
        max_mu_scale=1.0,
    ):
        super().__init__()
        self.d_model = d_model
        self.seq_len = seq_len
        self.sharpness = sharpness 
        self.cross_sharpness = cross_sharpness
        self.m_sharpness = m_sharpness
        self.mu_sharpness = mu_sharpness
        self.version = version

        self.max_mass_scale = max_mass_scale
        self.max_mu_scale = max_mu_scale
        
        # ============================================================
        # STEP 1: CONTINUOUS EMBEDDING & ENCODER
        # Math: z_t = W_in * x_t + b_in
        # W_in and b_in are TRAINABLE parameters. They learn how to 
        # map raw kinematics into a high-dimensional physical feature space.
        # ============================================================
        self.input_proj = nn.Linear(input_dim, d_model)
        self.input_norm = nn.LayerNorm(d_model)
        self.pos_encoder = PositionalEncoding(d_model, max_len=seq_len + 10)
        
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead, 
            dim_feedforward=dim_feedforward, 
            batch_first=True, dropout=dropout
        )
        self.transformer_encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_encoder_layers)

        # ============================================================
        # STEP 2: DECODER (GENERATING h_t_dec)
        # Math: h_t_dec = CrossAttention(q_t_query, H_enc, H_enc)
        # h_t_dec is the hidden state for time step 't' that integrates 
        # the entire motion context into a force-ready representation.
        # ============================================================
        
        if not self.version in [3, 4, 5, 6, 7, 8]:
            self.time_queries = nn.Parameter(torch.randn(1, seq_len, d_model))
        
        if not self.version in [3, 4, 5, 6, 7, 8]:
            self.cross_attn = nn.MultiheadAttention(d_model, nhead, batch_first=True, dropout=dropout)
            
            self.norm_dec = nn.LayerNorm(d_model)
            self.ffn_dec = nn.Sequential(
                nn.Linear(d_model, dim_feedforward), 
                nn.ReLU(), 
                nn.Linear(dim_feedforward, d_model)
            )
            self.norm_ffn = nn.LayerNorm(d_model)

        # ============================================================
        # STEP 3: INTERMEDIATE FORCE SEQUENCES
        # ============================================================
        self.net_force_mlp = nn.Sequential(
            nn.Linear(d_model, d_model), nn.ReLU(), nn.Linear(d_model, d_model)
        ) 
        self.fric_force_mlp = nn.Sequential(
            nn.Linear(d_model, d_model), nn.ReLU(), nn.Linear(d_model, d_model)
        ) 

        # Dedicated layers to project latent features into Newtons (PINN loss)
        if self.version in [1, 2, 3, 4, 5, 6, 7, 8]:
            self.phys_net_proj = nn.Linear(d_model, 1)
            self.phys_fric_proj = nn.Linear(d_model, 1)
        elif self.version == 2.5:
            self.phys_net_proj = nn.Linear(d_model * 2, 1)
            self.phys_fric_proj = nn.Linear(d_model * 2, 1)

        # ============================================================
        # STEP 4: GLOBAL READOUTS (MASS & MU)
        # ============================================================
        
        # --- A. MASS ESTIMATION ---
        if self.version in [1, 3, 4, 6]:
            self.q_mass = nn.Parameter(torch.randn(1, 1, d_model))
            self.mass_attn = nn.MultiheadAttention(d_model, 1, batch_first=True)
            self.mass_pred_mlp = nn.Sequential(
                nn.Linear(d_model, 64), 
                nn.ReLU(), 
                nn.Linear(64, 1),
                nn.Softplus() # Ensures mass > 0
            )
        elif self.version in [2, 2.5]:
            self.q_mass = nn.Parameter(torch.randn(1, 1, d_model * 2))
            self.mass_attn = nn.MultiheadAttention(d_model * 2, 1, batch_first=True) # Double dimension
            self.mass_pred_mlp = nn.Sequential(
                nn.Linear(d_model * 2, 64), 
                nn.ReLU(), 
                nn.Linear(64, 1),
                nn.Softplus() 
            )
        elif self.version == 5:
            self.q_mass = nn.Parameter(torch.randn(1, 1, d_model))
            self.mass_attn = nn.MultiheadAttention(d_model, 1, batch_first=True)
            self.mass_pred_mlp = nn.Sequential(
                nn.Linear(d_model, 128), 
                nn.GELU(), 
                nn.Linear(128, 1),
                nn.Softplus()
            )
        elif self.version == 7:
            self.q_mass = nn.Parameter(torch.randn(1, 1, d_model))
            self.mass_attn = nn.MultiheadAttention(d_model, 1, batch_first=True)
            self.mass_pred_mlp = nn.Sequential(
                nn.Linear(d_model, 64), 
                nn.GELU(), 
                nn.Linear(64, 1),
                nn.Softplus()
            )
        elif self.version == 8:
            self.q_mass = nn.Parameter(torch.randn(1, 1, d_model))
            self.mass_attn = nn.MultiheadAttention(d_model, 1, batch_first=True)
            self.mass_pred_mlp = nn.Sequential(
                nn.Linear(d_model, 128), 
                nn.GELU(), 
                nn.Linear(128, 1),
                nn.Sigmoid() # <-- Sigmoid bounded to block AdamW decay
            )
            # Prevent Initialization Shock
            # Sigmoid(-1.6) * 3.0 ≈ 0.5 kg. This starts the network at the dataset average!
            nn.init.constant_(self.mass_pred_mlp[2].bias, -1.6)

        # --- B. FRICTION ESTIMATION ---
        if self.version in [1, 3]:
            self.q_fric = nn.Parameter(torch.randn(1, 1, d_model))
            self.fric_attn = nn.MultiheadAttention(d_model, 1, batch_first=True)
            self.mu_pred_mlp = nn.Sequential(
                nn.Linear(d_model + 1, 64), 
                nn.ReLU(), 
                nn.Linear(64, 1),
                nn.Softplus() 
            )
        elif self.version in [2, 2.5]:
            self.q_fric = nn.Parameter(torch.randn(1, 1, d_model * 2))
            self.fric_attn = nn.MultiheadAttention(d_model * 2, 1, batch_first=True) 
            self.mu_pred_mlp = nn.Sequential(
                nn.Linear(d_model * 2 + 1, 64), 
                nn.ReLU(), 
                nn.Linear(64, 1),
                nn.Softplus() 
            )
        elif self.version == 4:
            self.q_fric = nn.Parameter(torch.randn(1, 1, d_model))
            self.fric_attn = nn.MultiheadAttention(d_model, 1, batch_first=True)
            self.mu_pred_mlp = nn.Sequential(
                nn.Linear(d_model, 64), 
                nn.ReLU(), 
                nn.Linear(64, 1),
                nn.Softplus() 
            )
        elif self.version == 5:
            self.q_fric = nn.Parameter(torch.randn(1, 1, d_model))
            self.fric_attn = nn.MultiheadAttention(d_model, 1, batch_first=True)
            self.mu_pred_mlp = nn.Sequential(
                nn.Linear(d_model, 128), 
                nn.GELU(), 
                nn.Linear(128, 1),
                nn.Sigmoid() 
            )
        elif self.version in [6, 7]:
            self.q_fric = nn.Parameter(torch.randn(1, 1, d_model))
            self.fric_attn = nn.MultiheadAttention(d_model, 1, batch_first=True)
            self.mu_pred_mlp = nn.Sequential(
                nn.Linear(d_model, 64), 
                nn.GELU(), 
                nn.Linear(64, 1),
                nn.Sigmoid()
            )
        elif self.version == 8:
            self.q_fric = nn.Parameter(torch.randn(1, 1, d_model))
            self.fric_attn = nn.MultiheadAttention(d_model, 1, batch_first=True)
            self.mu_pred_mlp = nn.Sequential(
                nn.Linear(d_model, 128), 
                nn.GELU(), 
                nn.Linear(128, 1),
                nn.Sigmoid()
            )
            # Prevent Friction Shock
            # Sigmoid(-0.85) * 1.0 ≈ 0.3. This starts friction safely.
            nn.init.constant_(self.mu_pred_mlp[2].bias, -0.85)

    def forward(self, extracted_vel):
        DEBUG_MODEL = False
        
        if len(extracted_vel.shape) == 2:
            x = extracted_vel.unsqueeze(-1)
        else:
            x = extracted_vel 
            
        B, T, _ = x.shape

        if DEBUG_MODEL:
            print("\n" + "="*60)
            print("🔍 PHYSICS TRANSFORMER INTERNAL DEBUGGER 🔍")
            print("="*60)
            print(f"1. Raw Input (x):       Mean {x.mean().item():.4f} | Std {x.std().item():.4f}")

        # 1. Project Input
        z = self.input_proj(x)
        
        # SKIP 1D NORM FOR VERSIONS 5 & 8 TO PREVENT MODE COLLAPSE
        if self.version in [5, 8]:
            z = z * math.sqrt(self.d_model)
        else:
            z = self.input_norm(z)
            z = z * math.sqrt(self.d_model)

        # 2. Add Positional Encoding
        z = self.pos_encoder(z)
        
        # 3. Transformer Encoder
        h_enc = self.transformer_encoder(z)
        
        # --- Decoding Logic ---
        if not self.version in [3, 4, 5, 6, 7, 8]:
            q_dec = self.time_queries.expand(B, -1, -1)
            attn_output, cross_weights_raw = self.cross_attn(
                query=q_dec*self.cross_sharpness, 
                key=h_enc, 
                value=h_enc, 
                average_attn_weights=False  
            )
            cross_weights = cross_weights_raw / (cross_weights_raw.sum(dim=-1, keepdim=True) + 1e-10)
            h_dec = self.norm_dec(q_dec + attn_output)
            h_dec = self.norm_ffn(h_dec + self.ffn_dec(h_dec))

        if self.version in [3, 4, 5, 6, 7, 8]:
            feat_net = self.net_force_mlp(h_enc)   
            feat_fric = self.fric_force_mlp(h_enc) 
        else:
            feat_net = self.net_force_mlp(h_dec)   
            feat_fric = self.fric_force_mlp(h_dec) 

        if self.version in [2, 2.5]:
            feat_net_enriched = torch.cat([feat_net, h_enc], dim=-1)
            feat_fric_enriched = torch.cat([feat_fric, h_enc], dim=-1)
        
        if self.version in [1, 2, 3, 4, 5, 6, 7, 8]:
            phys_net = self.phys_net_proj(feat_net)   
            phys_fric = self.phys_fric_proj(feat_fric) 
            phys_net = torch.abs(phys_net)
            phys_fric = torch.abs(phys_fric)
        elif self.version == 2.5:
            phys_net = self.phys_net_proj(feat_net_enriched)   
            phys_fric = self.phys_fric_proj(feat_fric_enriched) 
            phys_net = torch.abs(phys_net)
            phys_fric = torch.abs(phys_fric)

        # GLOBAL PROPERTY READOUT: Mass Prediction
        q_m_batch = self.q_mass.expand(B, -1, -1)
        if self.version in [1, 3, 4, 5, 6, 7, 8]:
            mass_ctx, mass_weights = self.mass_attn(query=q_m_batch*self.m_sharpness, key=feat_net, value=feat_net)
        elif self.version in [2, 2.5]:
            mass_ctx, mass_weights = self.mass_attn(query=q_m_batch*self.m_sharpness, key=feat_net_enriched, value=feat_net_enriched)
        
        raw_mass_pred = self.mass_pred_mlp(mass_ctx.squeeze(1))
        mass_pred = raw_mass_pred * self.max_mass_scale

        # GLOBAL PROPERTY READOUT: Friction Prediction
        q_f_batch = self.q_fric.expand(B, -1, -1)
        if self.version in [1, 3, 4, 5, 6, 7, 8]:
            fric_ctx, fric_weights = self.fric_attn(query=q_f_batch*self.mu_sharpness, key=feat_fric, value=feat_fric)
        elif self.version in [2, 2.5]:
            fric_ctx, fric_weights = self.fric_attn(query=q_f_batch*self.mu_sharpness, key=feat_fric_enriched, value=feat_fric_enriched)

        if self.version in [4, 5, 6, 7, 8]:
            fric_input = torch.cat([fric_ctx.squeeze(1)], dim=-1)
        else:
            fric_input = torch.cat([fric_ctx.squeeze(1), mass_pred], dim=-1)
            
        raw_mu_pred = self.mu_pred_mlp(fric_input)
        mu_pred = raw_mu_pred * self.max_mu_scale

        preds = torch.cat([mass_pred, mu_pred], dim=-1)

        if self.version in [3, 4, 5, 6, 7, 8]:
            return preds, (None, mass_weights, fric_weights), (phys_net, phys_fric)
        else:
            return preds, (cross_weights, mass_weights, fric_weights), (phys_net, phys_fric)

In [7]:
class PinnLossCalculator:
    def __init__(self, criterion, gravity=9.81):
        # CRITICAL: criterion MUST be initialized with reduction='none' 
        # e.g., nn.MSELoss(reduction='none')
        self.criterion = criterion
        self.g = gravity

    def net_force_law(self, mass, acceleration, target_force):
        """Law: F = m * a"""
        theory = mass * acceleration
        # Returns shape: [Batch, TimeSteps]
        return self.criterion(target_force, theory)

    def friction_law(self, mass, mu, robot_fz, target_fric):
        """Law: F_fric = mu * (m*g - Fz_robot)"""
        normal_force = (mass * self.g) - robot_fz
        theory = mu * torch.clamp(normal_force, min=0.0)
        # Returns shape: [Batch, TimeSteps]
        return self.criterion(target_fric, theory)

    def friction_direct_law(self, mu, normal_force_sensor, target_fric):
        """Law: F_fric = mu * F_normal_measured"""
        theory = mu * torch.clamp(normal_force_sensor, min=0.0)
        # Returns shape: [Batch, TimeSteps]
        return self.criterion(target_fric, theory)
    
    def inertia_consistency_law(self, mass_est, acceleration_raw, net_f_est):
        """PINN Net 4 Law: m = F_net / a (Ratio Consistency)"""
        # Calculate implied mass for every frame
        implied_mass = net_f_est / (acceleration_raw + 1e-6)
        
        # Expand mass_est [B, 1] to match [B, TimeSteps]
        mass_est_expanded = mass_est.expand_as(implied_mass)
        
        # Returns shape: [Batch, TimeSteps]. 
        # (The external apply_mask will ignore the division-by-zero spikes when a is near 0)
        return self.criterion(mass_est_expanded, implied_mass)

    def inertia_acceleration_law(self, mass_est, net_f_raw, acc_raw):
        """PINN Law: a = F_net / m"""
        eps = 1e-6
        acc_theory = net_f_raw / (mass_est + eps)
        # Returns shape: [Batch, TimeSteps]
        return self.criterion(acc_theory, acc_raw)

    def friction_decoupled_law(self, mass_est, mu_est, robot_fz, fric_f_est):
        """PINN Fric 5 Law: F_fric + mu*Fz_robot = mu*m*g"""
        effective_fric = fric_f_est + (mu_est * robot_fz)
        weight_ceiling = mu_est * mass_est * self.g
        # Returns shape: [Batch, TimeSteps]
        return self.criterion(effective_fric, weight_ceiling.expand_as(effective_fric))

    def net_force_law_smoothed(self, mass_est, acc_raw, net_f_est):
        """
        Improved PINN Net 4: Ratio of Means.
        Calculates one ratio for the whole window, but expands back to [Batch, TimeSteps]
        so the external apply_mask doesn't crash.
        """
        T = acc_raw.size(1)
        
        # Sum across the time window: [Batch, 1]
        sum_f = torch.sum(net_f_est, dim=1, keepdim=True)
        sum_a = torch.sum(acc_raw, dim=1, keepdim=True)
        
        # Implied mass for the window: [Batch, 1]
        implied_mass_window = sum_f / (sum_a + 1e-6)
        
        # Expand both to [Batch, TimeSteps]
        return self.criterion(mass_est.expand(-1, T), implied_mass_window.expand(-1, T))

    def friction_law_robust(self, mass_est, mu_est, robot_fz, fric_f_est):
        """
        Improved PINN Fric 5: Averages the linear balance PER SAMPLE.
        """
        T = robot_fz.size(1)
        
        # Calculate the balance for all time steps: [Batch, TimeSteps]
        left_side_seq = fric_f_est + (mu_est * robot_fz)
        
        # Average across the time dimension: [Batch, 1]
        left_side_mean = left_side_seq.mean(dim=1, keepdim=True)
        
        # Right side is constant for the window: [Batch, 1]
        right_side = mu_est * mass_est * self.g
        
        # Expand both to [Batch, TimeSteps] to match the mask
        return self.criterion(left_side_mean.expand(-1, T), right_side.expand(-1, T))

    def kinematic_consistency(self, mass_est, mu_est, push_f_lateral, push_f_vertical, acc_raw):
        """Calculates acceleration based on lateral push and vertical load."""
        eps = 1e-6
        normal_force = (mass_est * self.g) - push_f_vertical
        normal_force = torch.clamp(normal_force, min=0.0)
        
        fric_force = mu_est * normal_force
        acc_theory = (push_f_lateral - fric_force) / (mass_est + eps)
        
        # Returns shape: [Batch, TimeSteps]
        return self.criterion(acc_theory, acc_raw)

    # ----------- position base -----------
    def kinematic_position_consistency(self, mass_est, mu_est, push_f_lat, push_f_vert, acc_raw, dt=0.01):
        """x_theory = double_integral( (F_push - mu*(mg + Fz)) / m )"""
        eps = 1e-6
        normal_force = torch.clamp((mass_est * self.g) - push_f_vert, min=0.0)
        acc_theory = (push_f_lat - (mu_est * normal_force)) / (mass_est + eps)
        
        vel_theory = torch.cumsum(acc_theory * dt, dim=1)
        pos_theory = torch.cumsum(vel_theory * dt, dim=1)
        
        vel_raw = torch.cumsum(acc_raw * dt, dim=1)
        pos_raw = torch.cumsum(vel_raw * dt, dim=1)
        
        # Returns shape: [Batch, TimeSteps]
        return self.criterion(pos_theory, pos_raw)

    def inertia_position_law(self, mass_est, net_f_raw, acc_raw, dt=0.01):
        """x_theory = double_integral( F_net / m )"""
        eps = 1e-6
        acc_theory = net_f_raw / (mass_est + eps)
        
        vel_theory = torch.cumsum(acc_theory * dt, dim=1)
        pos_theory = torch.cumsum(vel_theory * dt, dim=1)
        
        vel_raw = torch.cumsum(acc_raw * dt, dim=1)
        pos_raw = torch.cumsum(vel_raw * dt, dim=1)
        
        # Returns shape: [Batch, TimeSteps]
        return self.criterion(pos_theory, pos_raw)

In [8]:
# ==========================================
# 5. OPTUNA OBJECTIVE FUNCTION
# ==========================================
def objective(trial):
    # --- A. Suggest Hyperparameters ---
    # Model Architecture
    d_model = trial.suggest_categorical("d_model", [32, 64, 128])
    num_enc = trial.suggest_int("num_enc", 2, 6)
    dropout = trial.suggest_float("dropout", 0.1, 0.5)
    
    # Sharpness
    m_sharpness = trial.suggest_float("m_sharpness", 1.0, 30.0)
    mu_sharpness = trial.suggest_float("mu_sharpness", 1.0, 30.0)
    cross_sharpness = trial.suggest_float("cross_sharpness", 1.0, 20.0)
    
    # Optimizer
    lr_optimizer = trial.suggest_categorical("lr_optimizer", ["Adam", "AdamW"])
    init_lr = trial.suggest_float("init_lr", 1e-4, 5e-3, log=True)

    transformer_ver = trial.suggest_int("transformer_ver", 4, 7)
    last_layer_ms = trial.suggest_float("last_layer_ms", 1.0, 5.0)
    last_layer_mus = 1.0

    mass_scale = trial.suggest_float("mass_scale", 1.0, 15.0)
    fric_scale = trial.suggest_float("fric_scale", 1.0, 15.0)
    
    # ==========================================
    # TUNABLE PINN PATTERNS (The 6 Combinations)
    # ==========================================
    pinn_pattern = trial.suggest_categorical("pinn_pattern", [
        "only_5", 
        "only_10", 
        "5_and_2-2_ann", 
        "10_and_9-2_ann", 
        "5_and_9-2_ann", 
        "10_and_2-2_ann"
    ])

    # First, initialize ALL coefficients and annealing to 0
    pinn_coeff_1 = pinn_coeff_2 = pinn_coeff_2_2 = pinn_coeff_3 = pinn_coeff_4 = 0.0
    pinn_coeff_5 = pinn_coeff_6 = pinn_coeff_7 = pinn_coeff_8 = pinn_coeff_9 = 0.0
    pinn_coeff_9_2 = pinn_coeff_9_3 = pinn_coeff_10 = pinn_coeff_11 = pinn_coeff_11_2 = 0.0
    pinn_coeff_annealing = 0

    # Next, dynamically apply the selected pattern!
    if pinn_pattern == "only_5":
        pinn_coeff_5 = trial.suggest_float("val_coeff_5", 5.0, 20.0)
        
    elif pinn_pattern == "only_10":
        pinn_coeff_10 = trial.suggest_float("val_coeff_10", 5.0, 20.0)
        
    elif pinn_pattern == "5_and_2-2_ann":
        pinn_coeff_5 = trial.suggest_float("val_coeff_5", 5.0, 20.0)
        pinn_coeff_2_2 = trial.suggest_float("val_coeff_2_2", 5.0, 10.0)
        pinn_coeff_annealing = 1
        
    elif pinn_pattern == "10_and_9-2_ann":
        pinn_coeff_10 = trial.suggest_float("val_coeff_10", 5.0, 20.0)
        pinn_coeff_9_2 = trial.suggest_float("val_coeff_9_2", 5.0, 10.0)
        pinn_coeff_annealing = 1
        
    elif pinn_pattern == "5_and_9-2_ann":
        pinn_coeff_5 = trial.suggest_float("val_coeff_5", 5.0, 20.0)
        pinn_coeff_9_2 = trial.suggest_float("val_coeff_9_2", 5.0, 10.0)
        pinn_coeff_annealing = 1
        
    elif pinn_pattern == "10_and_2-2_ann":
        pinn_coeff_10 = trial.suggest_float("val_coeff_10", 5.0, 20.0)
        pinn_coeff_2_2 = trial.suggest_float("val_coeff_2_2", 5.0, 10.0)
        pinn_coeff_annealing = 1

    # ==========================================
    # Fixed Configuration (from your original script)
    # ==========================================
    loss_type = "pinn" 
    task_coeff = 0.0
    force_coeff = 0.0
    acc_filter_threshold = 0.3
    vel_filter_threshold = 0.01
    
    # Annealing constants (Only matters if pinn_coeff_annealing == 1)
    annealing_start_epoch = 300 
    ramp_duration = 600   
    diff_coeffs_pinn4 = 0

    c_entropy_coeff = m_entropy_coeff = f_entropy_coeff = 0.0
    c_head_coeffs = torch.tensor([c_entropy_coeff] * 4).to(device) 

    
    # --- B. Initialization ---
    model = PhysicsTransformerEstimator(
        input_dim=1, d_model=d_model, nhead=4, num_encoder_layers=num_enc, 
        seq_len=SEQ_LEN, dropout=dropout, sharpness=1.0, cross_sharpness=cross_sharpness,
        m_sharpness=m_sharpness, mu_sharpness=mu_sharpness, version=transformer_ver, 
        max_mass_scale=last_layer_ms, max_mu_scale=last_layer_mus
    ).to(device)

    if lr_optimizer == "Adam":
        optimizer = optim.Adam(model.parameters(), lr=init_lr, weight_decay=1e-3)
    else:
        optimizer = optim.AdamW(model.parameters(), lr=init_lr, weight_decay=1e-3)
        
    scheduler = optim.lr_scheduler.OneCycleLR(
        optimizer, max_lr=init_lr, steps_per_epoch=len(train_loader), epochs=TUNE_EPOCHS, pct_start=0.1
    )
    
    task_cri = log_mse_loss
    force_cri = log_mse_loss
    pinn_cri = nn.L1Loss(reduction='none')
    phys = PinnLossCalculator(pinn_cri)

    # --- C. Training Loop ---
    for epoch in range(TUNE_EPOCHS):
        
        if pinn_coeff_annealing == 1:
            if epoch < annealing_start_epoch:
                current_pinn_coeff = 0.0
            else:
                ramp = min(1.0, (epoch - annealing_start_epoch) / ramp_duration)
                current_pinn_coeff = ramp
        else:
            current_pinn_coeff = 1.0

        model.train()
        for b_acc, b_vel, b_y, b_robot_fz, b_rhs_acc, b_lhs_net_f, b_table_fz, b_start_t, b_robot_fx in train_loader:
            b_vel, b_y = b_vel.to(device), b_y.to(device)
            b_robot_fz, b_rhs_acc = b_robot_fz.to(device), b_rhs_acc.to(device)
            b_lhs_net_f, b_table_fz = b_lhs_net_f.to(device), b_table_fz.to(device)
            b_robot_fx = b_robot_fx.to(device)
            
            optimizer.zero_grad()
            output, (c_weights, m_weights, f_weights), (net_f_est, fric_f_est) = model(b_vel)

            m_gt, mu_gt = b_y[:, 0].unsqueeze(1), b_y[:, 1].unsqueeze(1)
            net_f_gt = b_lhs_net_f
            normal_f_gt = torch.abs(b_table_fz) 
            fric_f_gt = mu_gt * torch.clamp(normal_f_gt, min=0.0)
            force_gt = torch.cat([net_f_gt, fric_f_gt], dim=-1)
            
            m_est, mu_est = output[:, 0].unsqueeze(1), output[:, 1].unsqueeze(1)
            net_f_est, fric_f_est = net_f_est.squeeze(-1), fric_f_est.squeeze(-1)
            force_est = torch.cat([net_f_est, fric_f_est], dim=-1)

            if loss_type == "data":        
                task_loss = task_cri(output, b_y) 
                weighted_task_loss = task_coeff * task_loss
                force_loss = force_cri(force_gt, force_est)
                weighted_force_loss = force_coeff * force_loss
                total_loss = weighted_task_loss + weighted_force_loss
                weighted_pinn_loss = torch.tensor(0.0).to(device)
                
            elif loss_type in ["pinn", "hybrid"]:
                is_accelerating_mask = (torch.abs(b_rhs_acc) > acc_filter_threshold).float()
                mask_net = is_accelerating_mask[:, :SEQ_LEN//2]
                
                is_sliding_mask = (torch.abs(b_vel.squeeze(-1)) > vel_filter_threshold).float()
                mask_fric = is_sliding_mask[:, SEQ_LEN//2:]
                
                def apply_mask(loss_tensor, mask):
                    active_frames = torch.clamp(mask.sum(), min=1.0)
                    return (loss_tensor * mask).sum() / active_frames

                # PINN calculations
                pinn_net_1 = apply_mask(phys.net_force_law(m_est, b_rhs_acc[:, :SEQ_LEN//2], net_f_est[:, :SEQ_LEN//2]), mask_net)
                pinn_net_2 = apply_mask(phys.net_force_law(m_est, b_rhs_acc[:, :SEQ_LEN//2], b_lhs_net_f[:, :SEQ_LEN//2]), mask_net)
                pinn_net_2_ann = pinn_net_2 
                pinn_net_3 = apply_mask(phys.net_force_law(m_gt, b_rhs_acc[:, :SEQ_LEN//2], net_f_est[:, :SEQ_LEN//2]), mask_net)
                pinn_net_4 = apply_mask(phys.net_force_law_smoothed(m_est, b_rhs_acc[:, :SEQ_LEN//2], net_f_est[:, :SEQ_LEN//2]), mask_net)
                pinn_net_4_ann = apply_mask(phys.net_force_law_smoothed(m_est, b_rhs_acc[:, :SEQ_LEN//2], b_lhs_net_f[:, :SEQ_LEN//2]), mask_net)
                pinn_acc_inertia = apply_mask(phys.inertia_acceleration_law(m_est, b_lhs_net_f[:, :SEQ_LEN//2], b_rhs_acc[:, :SEQ_LEN//2]), mask_net)
                pinn_pos_inertia = apply_mask(phys.inertia_position_law(m_est, b_lhs_net_f[:, :SEQ_LEN//2], b_rhs_acc[:, :SEQ_LEN//2]), mask_net)

                pinn_fric_1 = apply_mask(phys.friction_law(m_est, mu_est, b_robot_fz[:, SEQ_LEN//2:], fric_f_est[:, SEQ_LEN//2:]), mask_fric)
                pinn_fric_2 = apply_mask(phys.friction_law(m_est, mu_est, b_robot_fz[:, SEQ_LEN//2:], fric_f_gt[:, SEQ_LEN//2:]), mask_fric)
                pinn_fric_3 = apply_mask(phys.friction_direct_law(mu_gt, torch.abs(b_table_fz[:, SEQ_LEN//2:]), fric_f_est[:, SEQ_LEN//2:]), mask_fric)
                pinn_fric_4 = apply_mask(phys.friction_law(m_gt, mu_est, b_robot_fz[:, SEQ_LEN//2:], fric_f_gt[:, SEQ_LEN//2:]), mask_fric)
                pinn_fric_5 = apply_mask(phys.friction_law_robust(m_est, mu_est, b_robot_fz[:, SEQ_LEN//2:], fric_f_est[:, SEQ_LEN//2:]), mask_fric)
                pinn_fric_5_ann = apply_mask(phys.friction_law_robust(m_est, mu_est, b_robot_fz[:, SEQ_LEN//2:], fric_f_gt[:, SEQ_LEN//2:]), mask_fric)
                pinn_acc_consistency = apply_mask(phys.kinematic_consistency(m_gt, mu_est, b_robot_fx[:, SEQ_LEN//2:], b_robot_fz[:, SEQ_LEN//2:], b_rhs_acc[:, SEQ_LEN//2:]), mask_fric)
                pinn_acc_consistency_2 = apply_mask(phys.kinematic_consistency(m_est, mu_est, b_robot_fx[:, SEQ_LEN//2:], b_robot_fz[:, SEQ_LEN//2:], b_rhs_acc[:, SEQ_LEN//2:]), mask_fric)
                pinn_pos_kinematic = apply_mask(phys.kinematic_position_consistency(m_gt, mu_est, b_robot_fx[:, SEQ_LEN//2:], b_robot_fz[:, SEQ_LEN//2:], b_rhs_acc[:, SEQ_LEN//2:]), mask_fric)
                pinn_pos_kinematic_2 = apply_mask(phys.kinematic_position_consistency(m_est, mu_est, b_robot_fx[:, SEQ_LEN//2:], b_robot_fz[:, SEQ_LEN//2:], b_rhs_acc[:, SEQ_LEN//2:]), mask_fric)
                
                # Aggregation
                pinn_loss_1 = (mass_scale * pinn_net_1) + (fric_scale * pinn_fric_1)
                pinn_loss_2 = ((mass_scale * pinn_net_2) + (fric_scale * pinn_fric_2)) * current_pinn_coeff if pinn_coeff_annealing == 1 else (mass_scale * pinn_net_2) + (fric_scale * pinn_fric_2)
                pinn_loss_3 = ((mass_scale * pinn_net_2) + (fric_scale * pinn_fric_2) + (mass_scale * pinn_net_3) + (fric_scale * pinn_fric_3))
                pinn_loss_4 = ((mass_scale * pinn_net_2) + (fric_scale * pinn_fric_2) + (mass_scale * pinn_net_3) + (fric_scale * pinn_fric_3) + (mass_scale * pinn_net_1) + (fric_scale * pinn_fric_1))
                pinn_loss_5 = (mass_scale * pinn_net_2) + (fric_scale * pinn_fric_4)
                pinn_loss_6 = ((mass_scale * pinn_net_2) + (fric_scale * pinn_fric_4) + (mass_scale * pinn_net_3) + (fric_scale * pinn_fric_3))
                pinn_loss_7 = ((mass_scale * pinn_net_2) + (fric_scale * pinn_fric_4) + (mass_scale * pinn_net_3) + (fric_scale * pinn_fric_3) + (mass_scale * pinn_net_1) + (fric_scale * pinn_fric_1))
                pinn_loss_8 = ((mass_scale * pinn_net_4_ann) + (fric_scale * pinn_fric_5_ann)) * current_pinn_coeff if pinn_coeff_annealing == 1 else (mass_scale * pinn_net_4) + (fric_scale * pinn_fric_5)
                pinn_loss_9 = (mass_scale * pinn_net_2) + (fric_scale * pinn_acc_consistency) 
                pinn_loss_10 = (mass_scale * pinn_acc_inertia) + (fric_scale * pinn_acc_consistency) 
                pinn_loss_11 = (mass_scale * pinn_pos_inertia) + (fric_scale * pinn_pos_kinematic)  
                
                if pinn_coeff_annealing == 1:
                    pinn_loss_2_2 = (fric_scale * pinn_fric_2) * current_pinn_coeff
                    pinn_loss_9_2 = (fric_scale * pinn_acc_consistency_2) * current_pinn_coeff
                    pinn_loss_11_2 = (fric_scale * pinn_pos_kinematic_2) * current_pinn_coeff
                else:
                    pinn_loss_2_2 = fric_scale * pinn_fric_2 
                    pinn_loss_9_2 = fric_scale * pinn_acc_consistency_2 
                    pinn_loss_11_2 = fric_scale * pinn_pos_kinematic_2 

                weighted_pinn_loss = (pinn_coeff_1 * pinn_loss_1 + pinn_coeff_2 * pinn_loss_2 + pinn_coeff_3 * pinn_loss_3 
                                      + pinn_coeff_4 * pinn_loss_4 + pinn_coeff_5 * pinn_loss_5 + pinn_coeff_6 * pinn_loss_6
                                      + pinn_coeff_7 * pinn_loss_7 + pinn_coeff_8 * pinn_loss_8 + pinn_coeff_9 * pinn_loss_9
                                      + pinn_coeff_2_2 * pinn_loss_2_2 + pinn_coeff_9_2 * pinn_loss_9_2
                                      + pinn_coeff_10 * pinn_loss_10 + pinn_coeff_11 * pinn_loss_11 + pinn_coeff_11_2 * pinn_loss_11_2)

                if loss_type == "pinn":
                    total_loss = weighted_pinn_loss
                elif loss_type == "hybrid":
                    task_loss = task_cri(output, b_y) 
                    total_loss = (task_coeff * task_loss) + (force_coeff * force_cri(force_gt, force_est)) + weighted_pinn_loss
                
            # Entropy Losses
            if c_weights is not None:
                c_ent_all = -torch.sum(c_weights * torch.log(c_weights + 1e-10), dim=-1) 
                weighted_c_entropy = torch.sum(c_ent_all.mean(dim=(0, 2)) * c_head_coeffs)
            else:
                weighted_c_entropy = torch.tensor(0.0, device=device)
            
            weighted_m_entropy = m_entropy_coeff * -torch.sum(m_weights * torch.log(m_weights + 1e-10), dim=-1).mean()
            weighted_f_entropy = f_entropy_coeff * -torch.sum(f_weights * torch.log(f_weights + 1e-10), dim=-1).mean()
            
            total_loss += weighted_c_entropy + weighted_m_entropy + weighted_f_entropy
            
            total_loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            scheduler.step()

        # --- D. Sim-Validation Pass (For Pruning) ---
        model.eval()
        val_loss_sum = 0.0
        with torch.no_grad():
            for b_acc, b_vel, b_y, *_ in val_loader:
                output, _, _ = model(b_vel.to(device))
                val_loss_sum += F.mse_loss(output, b_y.to(device)).item() * b_vel.size(0)
                
        epoch_val_loss = val_loss_sum / len(val_idx)

        trial.report(epoch_val_loss, epoch)
        if trial.should_prune():
            raise optuna.exceptions.TrialPruned()

    # --- E. OOD Real-World Evaluation ---
    model.eval()
    all_mass_errs, all_mu_errs = [], []
    condition_folders = [f.path for f in os.scandir(OFFLINE_DATA_DIR) if f.is_dir()]
    
    with torch.no_grad():
        for folder in condition_folders:
            gt_path = os.path.join(folder, "ground_truth.csv")
            if not os.path.exists(gt_path): continue
                
            gt_df = pd.read_csv(gt_path)
            gt_dict = dict(zip(gt_df['Parameter'], gt_df['Value']))
            m_gt = float(gt_dict.get('M_GT', -1))
            raw_mu = gt_dict.get('MU_GT', 'None')
            mu_gt = float(raw_mu) if raw_mu not in ['None', 'N/A', 'NaN'] else np.nan

            for file_path in glob.glob(os.path.join(folder, "*_100steps.csv")):
                df_rw = pd.read_csv(file_path)
                df_rw['v_y_smoothed'] = df_rw['v_y'].rolling(window=SMOOTHING_WINDOW_SIZE, min_periods=1).mean()
                df_inf = df_rw[df_rw['is_inference_region'] == 1].copy()
                
                if len(df_inf) != SEQ_LEN: continue
                    
                X_vel_rw = torch.tensor(df_inf['v_y_smoothed'].values).unsqueeze(0).unsqueeze(-1).float().to(device)
                preds, _, _ = model(X_vel_rw)
                
                m_est = preds[0, 0].item()
                mu_est = preds[0, 1].item()
                
                all_mass_errs.append(abs(m_est - m_gt) / GLOBAL_M_RANGE)
                if not np.isnan(mu_gt):
                    all_mu_errs.append(abs(mu_est - mu_gt) / GLOBAL_MU_RANGE)

    avg_mass_nmae = np.mean(all_mass_errs) if all_mass_errs else float('inf')
    avg_mu_nmae = np.mean(all_mu_errs) if all_mu_errs else float('inf')
    
    combined_ood_error = avg_mass_nmae + avg_mu_nmae 

    if combined_ood_error < trial.study.user_attrs.get("best_score", float('inf')):
        trial.study.set_user_attr("best_score", combined_ood_error)
        torch.save(model.state_dict(), BEST_MODEL_SAVE_PATH)

    return combined_ood_error

In [9]:
# ==========================================
# 6. RUN OPTIMIZATION & VISUALIZE
# ==========================================
if __name__ == "__main__":
    # 1. Define the database file name
    db_url = "sqlite:///phypush_optuna.db"
    
    study = optuna.create_study(
        direction="minimize", 
        study_name="PhyPush_PINN",
        storage=db_url,         # <-- NEW: Save to database
        load_if_exists=True,    # <-- NEW: Resume if you stop and restart the cell
        pruner=optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=30)
    )

    print(f"🚀 Starting Optuna. Monitoring available at database: {db_url}")
    study.optimize(objective, n_trials=500)

    print("\n" + "="*50)
    print("🏆 OPTIMIZATION FINISHED")
    print("="*50)
    print("Best trial Combined OOD Error:", study.best_trial.value)
    print("Best hyperparameters:")
    for key, value in study.best_trial.params.items():
        print(f"  {key}: {value}")
    print(f"✅ Best model weights saved to: {BEST_MODEL_SAVE_PATH}")

[I 2026-03-04 01:15:14,976] A new study created in RDB with name: PhyPush_PINN


🚀 Starting Optuna. Monitoring available at database: sqlite:///phypush_optuna.db


[I 2026-03-04 01:24:49,421] Trial 0 finished with value: 1.0473336929047523 and parameters: {'d_model': 32, 'num_enc': 3, 'dropout': 0.4110331873353349, 'm_sharpness': 9.37489675646699, 'mu_sharpness': 8.093568897743598, 'cross_sharpness': 4.767014458308575, 'lr_optimizer': 'Adam', 'init_lr': 0.002623482562118182, 'transformer_ver': 4, 'last_layer_ms': 4.422698211717016, 'mass_scale': 9.409327189053572, 'fric_scale': 10.581024095399773, 'pinn_pattern': 'only_5', 'val_coeff_5': 12.929124350481684}. Best is trial 0 with value: 1.0473336929047523.
[I 2026-03-04 01:34:42,252] Trial 1 finished with value: 0.985130816225093 and parameters: {'d_model': 128, 'num_enc': 4, 'dropout': 0.3601251928799297, 'm_sharpness': 27.15038946278435, 'mu_sharpness': 12.150878065105212, 'cross_sharpness': 10.74509194032068, 'lr_optimizer': 'AdamW', 'init_lr': 0.002826380366961189, 'transformer_ver': 4, 'last_layer_ms': 1.1325161743870447, 'mass_scale': 4.273108797750832, 'fric_scale': 8.305098079543765, 'pinn

KeyboardInterrupt: 